# The Functional API

The Graph API (`StateGraph`) builds a workflow out of nodes and edges. LangGraph also has a Functional API that builds the same kind of workflow out of plain Python functions: `@task` for a unit of work, `@entrypoint` for the workflow that calls those units. This notebook builds a small essay-writing workflow with the Functional API: write an outline, write each section in parallel, then pause for a human to approve the draft.

In [ ]:
import time

from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

## Part 1: Tasks Are Functions, Not Nodes

A `@task` function is a normal Python function with one difference: calling it does not run it immediately and return a value. It schedules the work and returns a future. You get the actual result by calling `.result()` on that future. This is what makes running several tasks in parallel possible: call all of them first, then collect results after.

In [ ]:
@task
def write_outline(topic: str) -> str:
    """Ask the LLM for a short list of section titles for an essay."""
    prompt = (
        f"List exactly 3 short section titles for a short essay about '{topic}'. "
        "One title per line. No numbering, no extra text."
    )
    response = llm.invoke(prompt)
    return response.content

## Part 2: Writing Sections in Parallel

`write_section` is a second task. Because tasks return futures instead of blocking, calling `write_section` three times in a row starts three LLM calls, and only `.result()` waits for them. That is the whole trick behind parallel tasks: no special syntax, just "call first, read results after."

In [ ]:
@task
def write_section(topic: str, section_title: str) -> str:
    """Ask the LLM to write one short paragraph for a section."""
    prompt = f"Write one short paragraph (2-3 sentences) for an essay about '{topic}', for the section '{section_title}'."
    response = llm.invoke(prompt)
    return response.content

## Part 3: The Entrypoint Ties the Tasks Together

`@entrypoint` marks the function that LangGraph runs as the workflow. Inside it, we call `write_outline` once, then call `write_section` once per section title *before* reading any of their results, so the section calls run in parallel. `interrupt(...)` pauses the workflow and hands a value out to whoever is running it, for a human to review.

In [ ]:
@entrypoint(checkpointer=InMemorySaver())
def essay_workflow(topic: str):
    outline_text = write_outline(topic).result()
    section_titles = [line.strip("- ").strip() for line in outline_text.splitlines() if line.strip()]

    # Call every section task first, then collect results: this is what makes
    # the three LLM calls below run in parallel instead of one after another.
    futures = [write_section(topic, title) for title in section_titles]
    sections = [f.result() for f in futures]

    approved = interrupt({"outline": section_titles, "sections": sections})
    if not approved:
        return {"approved": False}

    essay = "\n\n".join(sections)
    return entrypoint.final(value={"approved": True, "essay": essay}, save=sections)

## Part 4: Running Until the Interrupt

A `thread_id` identifies one run of the workflow, the same way it identifies one conversation in the chatbot notebooks. Streaming the workflow runs it up to the `interrupt()` call and stops there; the interrupt's value is what a human reviewer would see.

In [ ]:
config = {"configurable": {"thread_id": "essay-1"}}

for chunk in essay_workflow.stream("the benefits of writing in plain language", config):
    print(chunk)

## Part 5: Resuming With a Human Decision

`Command(resume=...)` sends the human's decision back into the paused `interrupt()` call, which then returns that value inside the entrypoint. The workflow does not start over from scratch conceptually, but its Python body does re-run from the top (see Part 6); only the parallel LLM calls are skipped because their results were already cached.

In [ ]:
result = essay_workflow.invoke(Command(resume=True), config)
print(result["essay"])

## Part 6: The Determinism Rule

On resume, LangGraph re-runs the entrypoint function from the beginning. If the entrypoint body called `random.random()` or `time.time()` directly, or hit an API directly instead of through a `@task`, it would get a *different* value on resume than it did the first time, and the workflow's state would become inconsistent. `@task` results are cached per run, so a resumed task call returns the same result instead of re-executing.

The rule that follows from this: **side effects and non-deterministic calls (time, random numbers, API calls, database writes) belong inside `@task`, never directly in the `@entrypoint` body.**

The next cell demonstrates this with counters instead of an LLM, so the effect is easy to see without spending an API call.

In [ ]:
counters = {"entry_runs": 0, "task_runs": 0}

@task
def counted_task(x: int) -> int:
    counters["task_runs"] += 1
    return x * 2

@entrypoint(checkpointer=InMemorySaver())
def demo_workflow(x: int):
    counters["entry_runs"] += 1
    doubled = counted_task(x).result()
    interrupt("pause for review")
    return doubled

demo_config = {"configurable": {"thread_id": "demo-1"}}

for chunk in demo_workflow.stream(5, demo_config):
    print(chunk)
print("after first run:", counters)

demo_workflow.invoke(Command(resume=True), demo_config)
print("after resume:", counters)

## Part 7: Graph API vs Functional API

Both APIs compile to the same underlying LangGraph runtime (same checkpointer support, same `interrupt`/`Command(resume=...)` mechanism, same streaming). They differ in how you describe the workflow.

| | Graph API (`StateGraph`) | Functional API (`entrypoint`/`task`) |
|---|---|---|
| Unit of work | A node function, added with `add_node` | A `@task` function, called directly |
| Control flow | Edges and conditional edges you wire explicitly | Plain Python: `if`, `for`, calling functions |
| Shared state | A typed state object every node reads and updates | Local variables in the entrypoint function |
| Parallel work | Multiple nodes reached from the same edge | Call several tasks before reading any `.result()` |
| Best fit | Workflows with branching, loops, or many possible paths between steps | Workflows that read like a normal script: mostly a sequence, with some parallel steps |

Neither API is strictly better. A graph makes branching and looping between many nodes easy to see. The Functional API makes a mostly-linear workflow with a few parallel steps easy to write, because you do not have to name every step as a node or draw every edge.

## Summary

- `@task` marks a function as a unit of work. Calling it returns a future; `.result()` gets the value.
- Calling several `@task` functions before reading any `.result()` runs them in parallel.
- `@entrypoint(checkpointer=...)` marks the workflow function. It needs a `thread_id` in the config, same as a graph with a checkpointer.
- `interrupt(value)` pauses the workflow and surfaces `value` to whoever is running it; `Command(resume=...)` sends a value back in and continues from the same point.
- `entrypoint.final(value=..., save=...)` returns `value` to the caller now, while saving `save` as the state for the next invocation of the same thread.
- On resume, the entrypoint's Python body re-runs from the top; only `@task` results are cached. Side effects and non-deterministic calls must live inside `@task`, not in the entrypoint body directly.